##Cell 1 — Install libraries


In [1]:
!pip -q install pymupdf sentence-transformers faiss-cpu pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 33.0 MB/s eta 0:00:00


##Cell 2 — Mount Drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/Jasmine_Documents"

Mounted at /content/drive


##Cell 3 — Create proper metadata (improved categories)


In [3]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Jasmine_Documents")

def category(name):
    n = name.lower()

    if any(k in n for k in ["chromosome","mutagen","genotype","floral biology","correlation","speciation"]):
        return "Breeding"

    if any(k in n for k in ["morphological","varieties","qualitative","visual flower quality"]):
        return "Variety"

    if any(k in n for k in ["pruning","paclobutrazol","defoliation","off season","flowering strategy"]):
        return "Pruning"

    if any(k in n for k in ["drip","irrigation","water stress","moisture"]):
        return "Irrigation"

    if any(k in n for k in ["fertigation","fertilizer","npk","nutrient","micronutrient","manure"]):
        return "Nutrition"

    if any(k in n for k in ["mite","thrips","bud worm","midge","pest"]):
        return "Pest"

    if any(k in n for k in ["virus","wilt","leaf spot","disease"]):
        return "Disease"

    if any(k in n for k in ["shelf life","packaging","storage","post harvest","export"]):
        return "PostHarvest"

    if any(k in n for k in ["volatile","essential oil","gc-ms","aroma","fragrance","co2 extraction"]):
        return "EssentialOil"

    if any(k in n for k in ["economics","marketing","adoption","yield gap"]):
        return "Economics"

    return "Cultivation"

rows=[]

for folder in ROOT.iterdir():
    if folder.is_dir() and folder.name.startswith("Jasminum"):
        for pdf in folder.glob("*.pdf"):
            rows.append({
                "filename":pdf.name,
                "species":folder.name.replace("_"," "),
                "category":category(pdf.name),
                "source":"KrishiKosh",
                "path":str(pdf)
            })

df=pd.DataFrame(rows)

meta_dir=ROOT/"metadata"
meta_dir.mkdir(exist_ok=True)

df.to_csv(meta_dir/"metadata.csv",index=False)

print("PDFs:",len(df))
df.head()

PDFs: 178


,filename,species,category,source,path
0,Studies on insect diversity in jasmine (Jasmin...,Jasminum sambac,Cultivation,KrishiKosh,/content/drive/MyDrive/Jasmine_Documents/Jasmi...
1,Molecular diversity and function of jasmintide...,Jasminum sambac,Cultivation,KrishiKosh,/content/drive/MyDrive/Jasmine_Documents/Jasmi...
2,Technology Adoption Behaviour of Jasmine Growe...,Jasminum sambac,Economics,KrishiKosh,/content/drive/MyDrive/Jasmine_Documents/Jasmi...
3,Photosynthetic response to water stress and ch...,Jasminum sambac,Irrigation,KrishiKosh,/content/drive/MyDrive/Jasmine_Documents/Jasmi...
4,A study on the effect of Zamzam water on the g...,Jasminum sambac,Cultivation,KrishiKosh,/content/drive/MyDrive/Jasmine_Documents/Jasmi...


##Cell 4 — Extract text + Chunk

In [4]:
import fitz
import pandas as pd
from tqdm import tqdm

meta = pd.read_csv(ROOT/"metadata"/"metadata.csv")

# -----------------------------
# Chunk Configuration
# -----------------------------
CHUNK_SIZE = 2000        # words per chunk
CHUNK_OVERLAP = 400      # overlapping words

chunks = []
chunk_id = 0

def split_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    out = []

    i = 0
    while i < len(words):
        out.append(" ".join(words[i:i+size]))
        i += size - overlap

    return out


for _, row in tqdm(meta.iterrows(), total=len(meta)):

    doc = fitz.open(row.path)

    text = ""
    for page in doc:
        text += page.get_text()

    for c in split_text(text):
        chunks.append({
            "chunk_id": chunk_id,
            "text": c,
            "filename": row.filename,
            "species": row.species,
            "category": row.category,
            "source": row.source
        })
        chunk_id += 1

chunk_df = pd.DataFrame(chunks)
chunk_df.to_csv(ROOT/"metadata"/"chunks.csv", index=False)

print("Chunks:", len(chunk_df))

100%|██████████| 178/178 [02:24<00:00,  1.23it/s]


Chunks: 1114


##Cell 5 — Create embeddings

In [5]:
from sentence_transformers import SentenceTransformer

model=SentenceTransformer("BAAI/bge-small-en-v1.5")

texts=chunk_df.text.tolist()

embeddings=model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

##Cell 6 — Build FAISS index

In [6]:
import faiss
import numpy as np

emb=np.array(embeddings).astype("float32")

index=faiss.IndexFlatIP(emb.shape[1])

index.add(emb)

faiss.write_index(index,str(ROOT/"metadata"/"faiss.index"))

chunk_df.to_csv(ROOT/"metadata"/"chunk_metadata.csv",index=False)

print("FAISS saved!")

FAISS saved!


##Cell 7 — Semantic Search

In [7]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Load FAISS index
index = faiss.read_index(str(ROOT/"metadata"/"faiss.index"))

# Load metadata
chunk_df = pd.read_csv(ROOT/"metadata"/"chunk_metadata.csv")


# ---------- Auto Category Detection ----------
def detect_category(query):
    q = query.lower()

    if any(x in q for x in ["pruning", "off season", "flowering", "bloom"]):
        return "Pruning"

    elif any(x in q for x in ["fertilizer", "fertigation", "npk", "micronutrient", "nutrient"]):
        return "Nutrition"

    elif any(x in q for x in ["drip", "irrigation", "water", "soil moisture"]):
        return "Irrigation"

    elif any(x in q for x in [
        "bud worm", "bud borer", "leaf web worm",
        "thrips", "mite", "aphid", "whitefly",
        "blossom midge", "pest", "insect"
    ]):
        return "Pest"

    elif any(x in q for x in ["virus", "leaf spot", "rust", "anthracnose", "disease"]):
        return "Disease"

    elif any(x in q for x in ["shelf life", "storage", "packaging", "transport", "post harvest"]):
        return "PostHarvest"

    return None


# ---------- Semantic Search ----------
def search(query, k=5, species="Jasminum sambac"):

    category = detect_category(query)

    q = model.encode([query], normalize_embeddings=True)
    q = np.array(q).astype("float32")

    scores, ids = index.search(q, 40)

    results = chunk_df.iloc[ids[0]].copy()
    results["score"] = scores[0]

    # Species filter
    results = results[results["species"] == species]

    # Category filter
    if category:
        results = results[results["category"] == category]

    # ---------- Keyword Boost ----------
    ql = query.lower()

    if "bud worm" in ql or "bud borer" in ql:
        boost = results["filename"].str.contains(
            "bio-efficacy|biorational|bud worm|bud borer",
            case=False, na=False
        )
        results.loc[boost, "score"] += 0.10

    # Sort and remove duplicates
    results = results.sort_values("score", ascending=False)
    results = results.drop_duplicates(subset=["filename"])

    return results.head(k)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
search("How can I extend the shelf life of Gundumalli flowers?")

,chunk_id,text,filename,species,category,source,score
157,157,~ 598 ~ International Journal of Chemical Stud...,Influence of postharvest treatments on quality...,Jasminum sambac,PostHarvest,KrishiKosh,0.767702
111,111,~ 265 ~ ISSN Print: 2617-4693 ISSN Online: 261...,Enhancing shelf life and freshness retention i...,Jasminum sambac,PostHarvest,KrishiKosh,0.742119
147,147,Int.J.Curr.Microbiol.App.Sci (2019) 8(9): 1724...,Packaging Technology for Extending Shelf Life ...,Jasminum sambac,PostHarvest,KrishiKosh,0.711725
183,183,~ 3069 ~ International Journal of Chemical Stu...,Effect of hexanal and boric acid on shelf life...,Jasminum sambac,PostHarvest,KrishiKosh,0.703287
151,151,Int.J.Curr.Microbiol.App.Sci (2018) 7(2): 494-...,Standardization of Eco-Friendly Retail Package...,Jasminum sambac,PostHarvest,KrishiKosh,0.700577


In [8]:
retrieved = search("How to control jasmine bud worm?", k=5)
print(retrieved[["score","filename"]])

        score                                           filename
978  0.814256  Biological Study of Jasmine Bud Worm, Hendecas...
954  0.811299  Evaluation of Biorational Compounds for the Ma...
76   0.747241  Crop diversification for sustainable managemen...
137  0.712054  Influence of abiotic factors on major insect a...
956  0.708272  Potential volatiles emitted from jasmine plant...


##Cell 8 — RAG Answer Generation


In [9]:
query = "How should I prune Gundumalli in October?"

docs = search(query, k=4)

context = "\n\n".join(docs["text"])

prompt = f"""
You are JasmineGPT, an agricultural expert.

Answer only from the provided research context.

Context:
{context}

Question:
{query}

Give a farmer-friendly answer and cite the paper names.
"""

In [12]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")





In [46]:
# ============================================================
# Phase 1.1 — Profile / memory-only statements
# ============================================================
def is_profile_statement(question: str) -> bool:
    """
    True only for explicit farmer profile/context statements.
    These statements update conversation memory and do NOT trigger RAG.
    """
    q = question.lower().strip()

    # Keep this intentionally narrow.
    # Do NOT include generic patterns like "I have ... jasmine",
    # because those can describe pest/disease problems and should
    # continue to the existing RAG.
    patterns = [
        r"^i grow ",
        r"^i am growing ",
        r"^i'm growing ",
        r"^my crop is ",
        r"^i cultivate ",
        r"^i am cultivating ",
        r"^i'm cultivating ",
        r"^my farm grows ",
        r"^my jasmine farm grows ",
    ]

    return any(re.search(pattern, q) for pattern in patterns)


def handle_profile_statement(question, conversation_id):
    """
    Store explicit farmer crop/species/cultivar context without
    performing any RAG retrieval.
    """
    conv = get_conversation(conversation_id)
    memory = conv.get("memory", {})
    entities = extract_entities(question)

    species = entities.get("species")
    cultivar = entities.get("cultivar")

    if species:
        memory["species"] = species

    if cultivar:
        memory["cultivar"] = cultivar

    conv["memory"] = memory

    # Persist through the existing conversation store API.
    update_memory(
        conversation_id,
        species=memory.get("species"),
        cultivar=memory.get("cultivar")
    )

    return (
        "Got it. I’ll remember that information for this conversation.",
        {
            "route": "MEMORY_UPDATE",
            "memory_updated": True,
            "memory": memory
        }
    )


In [50]:
import requests
import os
def ask_jasmine_gpt(question, conversation_id, k=5):
  # Profile / memory-only statements
  # NOTE: The 'conversation_id' variable is not explicitly passed to this function.
  # If this function is called directly, 'conversation_id' will cause a NameError
  # unless it's defined in a global scope or passed via a wrapper.
  # This block likely belongs in a higher-level routing function that handles conversation context.
  if is_profile_statement(question):
      answer, info = handle_profile_statement(
          question,
          conversation_id
      )

      print("\n🌸 JasmineGPT")
      print("ROUTE = MEMORY_UPDATE")
      print("ANSWER:", answer)
      print("MEMORY:", info["memory"])

      return answer

  retrieved = search(question, k)
  retrieved = retrieved[retrieved["score"] > 0.70]
  retrieved = retrieved.drop_duplicates(subset=["filename"])
  if len(retrieved) == 0:
    print("🌸 JasmineGPT\n")
    print("The available research does not provide sufficient evidence.")
    return

  # Build context for LLM
  context = ""
  for _, row in retrieved.iterrows():
      context += f"""

PAPER: {row['filename']}
CATEGORY: {row['category']}

{row['text']}
"""
    # ---------- Prompt ----------
  prompt = f"""
You are JasmineGPT, an agricultural AI assistant.

Use ONLY the research context below.Answer ONLY from the retrieved research context.

If the retrieved documents do not contain the requested information,
reply: "The available research does not provide sufficient evidence."

Do not use external knowledge or guess values.
Always cite the retrieved paper titles.
 If a concentration, temperature, dosage or percentage is mentioned in the research context, reproduce it exactly. Never estimate or modify numerical values.

{context}

Question:
{question}


Instructions:
- Answer only from the retrieved research context.
- Never invent pesticide doses or fertilizer quantities.
- If the exact dosage is not mentioned in the papers, explicitly say "The retrieved papers do not specify the dosage."
- Cite the paper names at the end.
"""

    # ---------- OpenRouter ----------
  response = requests.post(
      "https://openrouter.ai/api/v1/chat/completions",
      headers={
          "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
          "Content-Type": "application/json"
      },
      json={
          "model":"openai/gpt-4.1-mini",
          "messages":[
              {"role":"system","content":"""
                   You are JasmineGPT, a Retrieval-Augmented agricultural assistant.

                  Rules:
                  - Use ONLY the retrieved research context.
                  - Never answer from your own knowledge.
                  - Never guess pesticide doses, fertilizer quantities, temperatures, or percentages.
                  - If evidence is missing, reply exactly:
                  'The available research does not provide sufficient evidence.'
                  - Always cite the retrieved paper titles used in the answer.
                 """},
              {"role":"user","content":prompt}
          ],
          "temperature":0.1,
          "max_tokens":700
      }
  )

  answer = response.json()["choices"][0]["message"]["content"]

    # ---------- Display ----------
  print("🌸 JasmineGPT\n")
  print(answer)

  print("\n" + "="*70)
  print("\n📚 Research Evidence")
  for i, (_, row) in enumerate(retrieved.iterrows(), 1):
    print(f"{i}. [{row['category']}] {row['filename']}  (Score: {row['score']:.3f})")

  return None

In [51]:
answer = ask_jasmine_gpt("A jasmine plant is producing plenty of new leaves and long vegetative shoots, but flower bud production has suddenly decreased. The leaves are dark green and appear healthy, while the farmer has been applying nitrogen fertilizer regularly. Soil moisture is adequate and there are no clearly visible insects on the leaves. Should the farmer apply more fertilizer to increase flowering, or could the current fertilizer practice itself be contributing to poor flowering? What should be checked before recommending any treatment?")

🌸 JasmineGPT

The observed condition of the jasmine plant—plenty of new leaves and long vegetative shoots but decreased flower bud production despite healthy dark green leaves and regular nitrogen fertilizer application—suggests that the current fertilizer practice might be contributing to poor flowering. Excessive nitrogen often promotes vegetative growth at the expense of reproductive growth (flowering). The research context highlights that micronutrients such as iron (Fe) and zinc (Zn) play essential roles in flowering and overall plant growth, including chlorophyll synthesis, photosynthesis, and hormone synthesis, which are critical for flower production and yield.

Before recommending any treatment, it is important to check the micronutrient status of the plant, particularly iron and zinc levels, as their deficiency or imbalance could limit flowering despite adequate nitrogen and soil moisture. The research shows that foliar application of FeSO4 (0.5%) and ZnSO4 (0.5%) twice after

Why are jasmine flower buds forming normally but repeatedly dropping before opening even though there are no visible pests, nutrient deficiency symptoms, or water stress?

Why are jasmine leaves curling and becoming distorted while the plant continues producing new shoots and no obvious insects are visible?

In [52]:
retrieved = search("How to control jasmine bud worm?", k=1)

print(retrieved.iloc[0]["filename"])
print(retrieved.iloc[0]["text"])

Biological Study of Jasmine Bud Worm, Hendecasis duplifascialis Hampsn. (Pyraustidae Lepidoptera) from India.pdf
Int.J.Curr.Microbiol.App.Sci (2018) 7(6): 2884-2888 2884 Original Research Article https://doi.org/10.20546/ijcmas.2018.706.339 Biological Study of Jasmine Bud Worm, Hendecasis duplifascialis Hampsn. (Pyraustidae: Lepidoptera) from India T. Krishna Chaitanya* and K. Kumar Department of Agricultural Entomology and Nematology, Pandit Jawaharlal Nehru College of Agriculture and Research Institute, Karaikal, U. T. of Puducherry – 609602, India *Corresponding author A B S T R A C T Introduction Jasmine (Jasminum sambac Aiton.) is one of the most important fragrant flower crops grown commercially for loose flowers. Jasmine buds are used for making garlands, bouquets, decorating women’s hair, for religious offerings and for the production of perfumed oils and attars. The flowers and other parts of the plant also find a place in useful medicines (Ramadas et al., 1985). The term jasm

In [53]:
search("How to control jasmine bud worm?", k=10)[
    ["score","filename","category"]
]

,score,filename,category
978,0.814256,"Biological Study of Jasmine Bud Worm, Hendecas...",Pest
954,0.811299,Evaluation of Biorational Compounds for the Ma...,Pest
76,0.747241,Crop diversification for sustainable managemen...,Pest
137,0.712054,Influence of abiotic factors on major insect a...,Pest
956,0.708272,Potential volatiles emitted from jasmine plant...,Pest
54,0.679551,"Bioecology of Jasmine Mite, Tetranychus urtica...",Pest
129,0.678004,Studies on pest complex and seasonal incidence...,Pest
128,0.672918,Seasonal Incidence of Major Insect and Mite Pe...,Pest


In [54]:
from sentence_transformers import SentenceTransformer

# Download from Hugging Face
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Save locally
model.save("models/bge-small-en-v1.5")

print("Model downloaded successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model downloaded successfully!


# FINAL INTEGRATED MODULE — Non-destructive Post-Harvest RAG

This section is an additive layer on top of the existing JasmineGPT notebook.
The existing FAISS index/search code is preserved for cultivation, pests, diseases, nutrition, irrigation, pruning, etc. A separate post-harvest index is built from the five verified production papers only.

**Production documents:** JAS-SAM-001, JAS-SAM-002, JAS-SAM-004, JAS-SAM-005, JAS-AUR-001.

**Excluded:** JAS-SAM-003 (Singh 2009) because the available scan was not reliable enough for production RAG ingestion. It remains an exclusion record only.


In [56]:
# ============================================================
# FINAL INTEGRATED POST-HARVEST MODULE — configuration/imports
# ============================================================
from pathlib import Path
import os, re, json, math, warnings
import pandas as pd
import numpy as np
import fitz
import faiss
from sentence_transformers import SentenceTransformer

ROOT = Path("/content/drive/MyDrive/Jasmine_Documents")
META_DIR = ROOT / "metadata"
PH_OUT = META_DIR / "final_postharvest_rag_v2"
PH_OUT.mkdir(parents=True, exist_ok=True)

POSTHARVEST_DOCS = [
    "JAS-SAM-001", "JAS-SAM-002", "JAS-SAM-004", "JAS-SAM-005", "JAS-AUR-001"
]
EXCLUDED_DOCS = {
    "JAS-SAM-003": "Excluded from production RAG: available scan has insufficient extraction quality for reliable automated evidence retrieval."
}

EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
PH_INDEX_PATH = PH_OUT / "postharvest_faiss.index"
PH_CHUNKS_PATH = PH_OUT / "postharvest_chunks.csv"
PH_META_PATH = PH_OUT / "postharvest_metadata_v2.csv"
PH_CONFIG_PATH = PH_OUT / "rag_config_v2.json"
PH_EVAL_PATH = PH_OUT / "retrieval_evaluation_v2.csv"
PH_QUALITY_PATH = PH_OUT / "pdf_extraction_quality_v2.csv"

print("ROOT:", ROOT)
print("Post-harvest output:", PH_OUT)
print("Production docs:", POSTHARVEST_DOCS)
print("Excluded:", EXCLUDED_DOCS)


ROOT: /content/drive/MyDrive/Jasmine_Documents
Post-harvest output: /content/drive/MyDrive/Jasmine_Documents/metadata/final_postharvest_rag_v2
Production docs: ['JAS-SAM-001', 'JAS-SAM-002', 'JAS-SAM-004', 'JAS-SAM-005', 'JAS-AUR-001']
Excluded: {'JAS-SAM-003': 'Excluded from production RAG: available scan has insufficient extraction quality for reliable automated evidence retrieval.'}


In [57]:
# ============================================================
# Load verified post-harvest metadata — never fabricate values
# ============================================================
def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

metadata_candidates = [
    META_DIR / "postharvest_metadata_FINAL_6_papers.csv",
    META_DIR / "final_postharvest_rag_v2" / "postharvest_metadata_v2_with_exclusion_log.csv",
    META_DIR / "postharvest_metadata_v2_with_exclusion_log.csv",
]
meta_path = find_first_existing(metadata_candidates)
if meta_path is None:
    raise FileNotFoundError(
        "Verified post-harvest metadata CSV was not found. Expected one of: " +
        ", ".join(str(x) for x in metadata_candidates)
    )

meta_all = pd.read_csv(meta_path).fillna("")
if "document_id" not in meta_all.columns:
    raise ValueError("Post-harvest metadata must contain document_id")

ph_meta = meta_all[meta_all["document_id"].isin(POSTHARVEST_DOCS)].copy()
missing = sorted(set(POSTHARVEST_DOCS) - set(ph_meta["document_id"]))
if missing:
    raise ValueError(f"Missing verified metadata for production documents: {missing}")

print("Loaded metadata:", meta_path)
print(ph_meta[[c for c in ["document_id","filename","title","species","category","decision"] if c in ph_meta.columns]].to_string(index=False))


Loaded metadata: /content/drive/MyDrive/Jasmine_Documents/metadata/postharvest_metadata_FINAL_6_papers.csv
document_id                                                       filename                                                                                                                       title              species                                category                   decision
JAS-SAM-001 Jawaharlal_2012_Packaging_technology_for_export_of_jasmine.pdf                                                   Packaging technology for export of jasmine (Jasminum sambac Ait.) flowers      Jasminum sambac packaging; transportation; post_harvest FINAL_INCLUDE_WITH_CONTEXT
JAS-SAM-002             Choudhury_2019_Packaging_Technology_Gundumalli.pdf                           Packaging Technology for Extending Shelf Life of Jasmine (Jasminum sambac CV. Gundumalli) Flowers      Jasminum sambac        packaging; storage; post_harvest FINAL_INCLUDE_WITH_CONTEXT
JAS-SAM-004                Chito

In [58]:
# ============================================================
# Resolve the exact five PDF files from Drive
# ============================================================
all_pdfs = {}
for species_dir in [ROOT / "Jasminum_sambac", ROOT / "Jasminum_auriculatum"]:
    if species_dir.exists():
        for p in species_dir.rglob("*.pdf"):
            all_pdfs[p.name] = p

resolved_rows=[]
unresolved=[]
for _, r in ph_meta.iterrows():
    fname = str(r.get("filename", "")).strip()
    p = all_pdfs.get(fname)
    if p is None:
        # Conservative fallback: filename stem tokens must strongly overlap.
        stem = Path(fname).stem.lower()
        candidates = [p for n,p in all_pdfs.items() if Path(n).stem.lower()==stem]
        if candidates:
            p = candidates[0]
    if p is None:
        unresolved.append((r["document_id"], fname))
    else:
        resolved_rows.append({"document_id":r["document_id"], "filename":fname, "path":str(p), "species":r["species"]})

if unresolved:
    print("UNRESOLVED PDFs:", unresolved)
    print("Available PDF count:", len(all_pdfs))
    raise FileNotFoundError("Could not resolve all five verified production PDFs. Check filenames in metadata.csv and Drive.")

resolution_df=pd.DataFrame(resolved_rows)
resolution_df.to_csv(PH_OUT/"pdf_resolution_v2.csv", index=False)
print(resolution_df.to_string(index=False))


document_id                                                       filename                                                                                                                    path              species
JAS-SAM-001 Jawaharlal_2012_Packaging_technology_for_export_of_jasmine.pdf /content/drive/MyDrive/Jasmine_Documents/Jasminum_sambac/Jawaharlal_2012_Packaging_technology_for_export_of_jasmine.pdf      Jasminum sambac
JAS-SAM-002             Choudhury_2019_Packaging_Technology_Gundumalli.pdf             /content/drive/MyDrive/Jasmine_Documents/Jasminum_sambac/Choudhury_2019_Packaging_Technology_Gundumalli.pdf      Jasminum sambac
JAS-SAM-004                Chitosan-based_biodegradable_packaging_2024.pdf                /content/drive/MyDrive/Jasmine_Documents/Jasminum_sambac/Chitosan-based_biodegradable_packaging_2024.pdf      Jasminum sambac
JAS-SAM-005                      Mycelium_foam_packaging_J_sambac_2026.pdf                      /content/drive/MyDrive/Jasmine_Documents

In [59]:
# ============================================================
# Fresh extraction + high-signal chunking for post-harvest only
# ============================================================
CHUNK_WORDS = 420
OVERLAP_WORDS = 80

def split_text_words(text, size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    words = text.split()
    if not words:
        return []
    step = max(1, size-overlap)
    return [" ".join(words[i:i+size]) for i in range(0, len(words), step)]

chunks=[]
quality=[]
for _, r in resolution_df.iterrows():
    path=Path(r.path)
    doc=fitz.open(path)
    page_texts=[]
    char_count=0
    nonempty_pages=0
    for page_no, page in enumerate(doc, start=1):
        txt=page.get_text("text") or ""
        txt=" ".join(txt.split())
        page_texts.append((page_no,txt))
        char_count += len(txt)
        nonempty_pages += int(bool(txt))
    quality.append({
        "document_id":r.document_id,
        "filename":r.filename,
        "pages":len(doc),
        "nonempty_text_pages":nonempty_pages,
        "extracted_chars":char_count,
        "extraction_status":"OK" if char_count>=500 else "LOW_TEXT_REVIEW"
    })
    chunk_no=0
    for page_no, txt in page_texts:
        for part in split_text_words(txt):
            if len(part.split()) < 30:
                continue
            chunks.append({
                "chunk_id":f"{r.document_id}-P{page_no}-C{chunk_no}",
                "document_id":r.document_id,
                "filename":r.filename,
                "species":r.species,
                "page":page_no,
                "text":part,
            })
            chunk_no += 1

doc_quality=pd.DataFrame(quality)
post_chunks=pd.DataFrame(chunks)
if post_chunks.empty:
    raise RuntimeError("No post-harvest chunks were extracted.")
post_chunks.to_csv(PH_CHUNKS_PATH,index=False)
doc_quality.to_csv(PH_QUALITY_PATH,index=False)
print("Chunks:",len(post_chunks))
print(doc_quality.to_string(index=False))


Chunks: 86
document_id                                                       filename  pages  nonempty_text_pages  extracted_chars extraction_status
JAS-SAM-001 Jawaharlal_2012_Packaging_technology_for_export_of_jasmine.pdf     10                   10            34193                OK
JAS-SAM-002             Choudhury_2019_Packaging_Technology_Gundumalli.pdf      9                    9            20495                OK
JAS-SAM-004                Chitosan-based_biodegradable_packaging_2024.pdf     11                   11            41516                OK
JAS-SAM-005                      Mycelium_foam_packaging_J_sambac_2026.pdf      8                    8            33315                OK
JAS-AUR-001             Sunny_2022_Pacha_Mullai_postharvest_treatments.pdf      6                    6            26698                OK


In [60]:
# ============================================================
# Build a separate post-harvest FAISS index
# ============================================================
ph_model = SentenceTransformer(EMBED_MODEL_NAME)
ph_embeddings = ph_model.encode(
    post_chunks["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)
ph_emb = np.asarray(ph_embeddings, dtype="float32")
ph_index = faiss.IndexFlatIP(ph_emb.shape[1])
ph_index.add(ph_emb)
faiss.write_index(ph_index, str(PH_INDEX_PATH))

print("Saved separate post-harvest index:", PH_INDEX_PATH)
print("Vector count:", ph_index.ntotal)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Saved separate post-harvest index: /content/drive/MyDrive/Jasmine_Documents/metadata/final_postharvest_rag_v2/postharvest_faiss.index
Vector count: 86


In [61]:
# ============================================================
# Verified document evidence profiles
# These are manually verified study facts, not model guesses.
# The profiles are used for routing/grounding when PDF wording varies.
# ============================================================
PROFILES = {
    "JAS-SAM-001": {
        "species": "Jasminum sambac",
        "cultivar": "NOT_REPORTED",
        "domains": {"packaging","transportation"},
        "concepts": {"export packaging","gel ice","thermocol","boric acid 4%","box a","reefer van"},
        "temperatures_c": [],
        "source_note": "Jawaharlal et al. (2012), packaging technology for export of J. sambac. Controlled packaging/chemical-treatment study; gel-ice and transport context reported.",
        "verified_facts": [
            "Best reported combination B1T4: 4% boric acid + Box A + thermocol outer packaging + intermittent gel ice.",
            "Box A was an aluminium-foil-lined cardboard box (14 × 11 × 14 cm).",
            "The package temperature was reported as 4.2°C initially and 16.5°C at 36 h.",
            "A reefer van and New Jersey/US export context were reported; this was not a controlled real-flight trial."
        ]
    },
    "JAS-SAM-002": {
        "species": "Jasminum sambac",
        "cultivar": "Gundumalli",
        "domains": {"packaging","storage"},
        "concepts": {"gundumalli","boric acid 4%","60 micron","60 µm","polythene bag","cold storage"},
        "temperatures_c": [7],
        "source_note": "Choudhury et al. (2019), direct J. sambac cv. Gundumalli packaging/shelf-life experiment.",
        "verified_facts": [
            "Polythene bags were 20 × 12 cm, heat sealed, with no vents reported.",
            "The tested bag thicknesses included 40 µm and 60 µm.",
            "The best reported combination was 4% boric acid + 60 µm polythene + 7°C cold storage.",
            "Cold-room condition was 7°C with 80–85% RH.",
            "At 24 h for the best treatment, freshness was 98.75% and colour retention was 100%."
        ]
    },
    "JAS-SAM-004": {
        "species": "Jasminum sambac",
        "cultivar": "NOT_REPORTED",
        "domains": {"packaging","storage"},
        "concepts": {"chitosan","biodegradable packaging","glycerol film","tween 80","cold storage"},
        "temperatures_c": [5],
        "source_note": "Ffadhilah et al. (2024), chitosan-based biodegradable packaging study on J. sambac.",
        "verified_facts": [
            "Storage was at 5°C with observations at 24, 48 and 72 h.",
            "Chitosan-glycerol and chitosan-Tween 80 films were tested.",
            "Reported shelf life was 4.16 d for the glycerol formulation and 3.80 d for the Tween 80 formulation.",
            "The study used a thermocol box (53 × 38 × 37 cm)."
        ]
    },
    "JAS-SAM-005": {
        "species": "Jasminum sambac",
        "cultivar": "Ramanathapuram Gundumalli",
        "domains": {"packaging","transportation"},
        "concepts": {"mycelium foam","long-distance transport","gel ice","pre-cooling","EPS"},
        "temperatures_c": [],
        "source_note": "M. Mohamed Asik et al. (2026), mycelium foam packaging during simulated long-distance transport of J. sambac.",
        "verified_facts": [
            "Packaging treatments included EPS, mycelium foam, coir-pith lined carton and aluminium-foil-lined carton.",
            "Mycelium foam was 2.5 cm thick with density 0.15 g/cm³ and thermal conductivity 0.04 W/mK.",
            "Flowers were pre-cooled with cold water at 4–6°C for 10–15 min and gel ice was used at a 1:2 product-to-gel-ice ratio.",
            "The 48 h test was a simulated long-distance export condition at 28–32°C and 60–70% RH after gel-ice packaging."
        ]
    },
    "JAS-AUR-001": {
        "species": "Jasminum auriculatum",
        "cultivar": "Pacha Mullai ecotype",
        "domains": {"packaging","storage"},
        "concepts": {"pacha mullai","boric acid 4%","sucrose 4%","polyethylene bag","cold storage"},
        "temperatures_c": [5],
        "source_note": "Sunny et al. (2022), post-harvest treatments for J. auriculatum ecotype Pacha Mullai.",
        "verified_facts": [
            "Polyethylene bags (reported as 15 × 10 cm, 200 gauge) were heat sealed; ventilation was not reported.",
            "Cold storage was 5°C with 80–85% RH.",
            "Treatments included 4% sucrose and 4% boric acid under room and refrigerated conditions.",
            "The best reported treatment was 4% boric acid + 5°C; reported shelf life was 177.85 h."
        ]
    }
}

# Configuration guard: profile IDs must match production list.
assert set(PROFILES) == set(POSTHARVEST_DOCS)


In [62]:
# ============================================================
# Query understanding / conservative post-harvest routing
# ============================================================
SPECIES_ALIASES = {
    "sambac": "Jasminum sambac", "gundumalli": "Jasminum sambac",
    "ramanathapuram gundumalli": "Jasminum sambac",
    "auriculatum": "Jasminum auriculatum", "pacha mullai": "Jasminum auriculatum"
}

POST_TERMS = {
    "harvest", "harvesting", "post-harvest", "postharvest", "packaging", "package", "packing",
    "storage", "store", "shelf life", "transport", "transportation", "long-distance transport",
    "export packaging", "gel ice", "cold storage", "freshness", "flower opening", "colour retention",
    "polythene", "polyethylene", "thermocol", "mycelium", "chitosan", "passive map"
}
GENERAL_NONPH_TERMS = {
    "pesticide", "insecticide", "fungicide", "fertilizer", "fertiliser", "fertigation", "npk",
    "bud worm", "bud borer", "thrips", "mite", "aphid", "whitefly", "disease", "virus",
    "leaf spot", "irrigation", "drip", "pruning", "paclobutrazol", "nutrient", "manure"
}

CONCEPT_HINTS = {
    "passive_map": ["passive map", "passive modified atmosphere"],
    "gundumalli": ["gundumalli"],
    "pacha_mullai": ["pacha mullai"],
    "mycelium": ["mycelium", "fungal foam"],
    "chitosan": ["chitosan", "glycerol film", "tween 80"],
    "boric_60um": ["4% boric acid", "boric acid and 60", "60 micron", "60 µm", "60 um"],
    "gel_ice": ["gel ice", "gel-ice", "ice packaging"],
    "export": ["export", "reefer", "new jersey", "air transport"],
}

def extract_species(q):
    ql=q.lower()
    hits=[]
    if "jasminum auriculatum" in ql or "auriculatum" in ql or "pacha mullai" in ql:
        hits.append("Jasminum auriculatum")
    if "jasminum sambac" in ql or "sambac" in ql or "gundumalli" in ql or "ramanathapuram" in ql:
        hits.append("Jasminum sambac")
    return list(dict.fromkeys(hits))

def extract_domains(q):
    ql=q.lower(); out=set()
    if any(x in ql for x in ["package","packaging","packing","polythene","polyethylene","chitosan","mycelium","gel ice","thermocol","60 micron"]): out.add("packaging")
    if any(x in ql for x in ["storage","store","shelf life","cold storage","temperature","°c","celsius"]): out.add("storage")
    if any(x in ql for x in ["transport","transportation","long-distance","export","reefer","air transport","gel ice"]): out.add("transportation")
    if any(x in ql for x in ["harvest","harvesting"]): out.add("harvesting")
    return sorted(out)

def extract_temperature(q):
    vals=re.findall(r"(-?\d+(?:\.\d+)?)\s*(?:°\s*)?(?:c|celsius)", q.lower())
    return [float(v) for v in vals]

def detect_post_concepts(q):
    ql=q.lower(); found=[]
    for key, phrases in CONCEPT_HINTS.items():
        if any(p in ql for p in phrases): found.append(key)
    return found

def is_probably_postharvest(q):
    ql=q.lower()
    has_ph=any(t in ql for t in POST_TERMS)
    has_nonph=any(t in ql for t in GENERAL_NONPH_TERMS)
    return has_ph and not (has_nonph and not any(x in ql for x in ["flower", "flowers", "shelf life", "packaging", "post-harvest", "postharvest"]))


def route_postharvest(question):
    ql=question.lower()
    species=extract_species(question)
    domains=extract_domains(question)
    concepts=detect_post_concepts(question)
    temps=extract_temperature(question)

    # Explicitly unsupported passive MAP: Singh is excluded from production.
    if "passive map" in ql or "passive modified atmosphere" in ql:
        return {"status":"INSUFFICIENT_EVIDENCE", "selected_docs":[], "species":species, "domains":domains, "concepts":concepts, "temperatures":temps, "reason":"The passive-MAP study was excluded from production RAG because its available scan was not reliable for extraction."}

    # A harvesting query cannot be answered from these five production papers.
    if "harvest" in ql or "harvesting" in ql:
        harvest_docs=[d for d,p in PROFILES.items() if "harvesting" in p["domains"]]
        if not harvest_docs:
            return {"status":"INSUFFICIENT_EVIDENCE", "selected_docs":[], "species":species, "domains":domains, "concepts":concepts, "temperatures":temps, "reason":"No included production paper provides direct harvesting-time evidence."}

    candidates=[]
    for doc_id,p in PROFILES.items():
        score=0
        if species and p["species"] in species: score += 5
        if domains:
            score += 3*len(set(domains)&p["domains"])
        if concepts:
            # concept aliases are stored as exact verified terms where possible
            for c in concepts:
                if c=="boric_60um" and ("boric acid 4%" in p["concepts"] and ("60 micron" in p["concepts"] or "60 µm" in p["concepts"])): score += 8
                elif c=="gundumalli" and "gundumalli" in p["concepts"]: score += 8
                elif c=="pacha_mullai" and "pacha mullai" in p["concepts"]: score += 8
                elif c=="mycelium" and "mycelium foam" in p["concepts"]: score += 8
                elif c=="chitosan" and "chitosan" in p["concepts"]: score += 8
                elif c=="gel_ice" and "gel ice" in p["concepts"]: score += 5
                elif c=="export" and "export packaging" in p["concepts"]: score += 5
        if temps:
            for t in temps:
                if any(abs(float(x)-t)<1e-9 for x in p["temperatures_c"]): score += 7
        candidates.append((score,doc_id))

    candidates.sort(reverse=True)
    selected=[]
    if candidates and candidates[0][0] >= 5:
        best_score=candidates[0][0]
        # Keep all strong direct matches for multi-study questions.
        for sc,doc_id in candidates:
            if sc >= max(5, best_score-3): selected.append(doc_id)

    # Generic storage-temperature question: return the direct storage studies.
    if "storage" in domains and not concepts and not temps and species:
        selected=[d for d,p in PROFILES.items() if p["species"] in species and "storage" in p["domains"]]

    if not selected:
        return {"status":"INSUFFICIENT_EVIDENCE", "selected_docs":[], "species":species, "domains":domains, "concepts":concepts, "temperatures":temps, "reason":"No included verified post-harvest document directly supports this request."}

    status="MULTIPLE_STUDIES" if len(selected)>1 else "DIRECT_EVIDENCE"
    return {"status":status,"selected_docs":selected,"species":species,"domains":domains,"concepts":concepts,"temperatures":temps,"reason":"Direct verified post-harvest evidence is available."}


In [63]:
# ============================================================
# Hybrid retrieval inside selected post-harvest documents only
# ============================================================
def lexical_score(query, text):
    q_tokens=set(re.findall(r"[a-z0-9µ°%.-]+", query.lower()))
    t=text.lower()
    hits=sum(1 for tok in q_tokens if len(tok)>2 and tok in t)
    return hits/max(1,len(q_tokens))

def retrieve_postharvest(question, selected_docs, k=6):
    if not selected_docs:
        return pd.DataFrame()
    qv=ph_model.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids=ph_index.search(qv, min(len(post_chunks), max(30,k*8)))
    rows=[]
    for sc,i in zip(scores[0], ids[0]):
        if i < 0: continue
        r=post_chunks.iloc[int(i)].copy()
        if r.document_id not in selected_docs: continue
        lex=lexical_score(question,r.text)
        r["semantic_score"]=float(sc)
        r["lexical_score"]=float(lex)
        r["score"]=0.75*float(sc)+0.25*float(lex)
        rows.append(r)
    if not rows: return pd.DataFrame()
    out=pd.DataFrame(rows).sort_values("score",ascending=False)
    # Prefer diversity, but keep up to two strong passages per selected document.
    chosen=[]; counts={}
    for _,r in out.iterrows():
        d=r.document_id
        if counts.get(d,0)>=2: continue
        chosen.append(r)
        counts[d]=counts.get(d,0)+1
        if len(chosen)>=k: break
    return pd.DataFrame(chosen)


def build_verified_context(route):
    blocks=[]
    for doc_id in route["selected_docs"]:
        p=PROFILES[doc_id]
        blocks.append(
            f"DOCUMENT {doc_id}\n"
            f"Species: {p['species']}\n"
            f"Cultivar/ecotype: {p['cultivar']}\n"
            f"Source note: {p['source_note']}\n"
            f"Verified facts:\n- " + "\n- ".join(p["verified_facts"])
        )
    return "\n\n".join(blocks)


In [64]:
# ============================================================
# Safe post-harvest answer layer
# ============================================================
def answer_postharvest(question, k=6, show_debug=True):
    route=route_postharvest(question)
    if show_debug:
        print("="*80)
        print("QUERY:", question)
        print("ROUTE:", route["status"])
        print("SPECIES:", route["species"])
        print("DOMAINS:", route["domains"])
        print("CONCEPTS:", route["concepts"])
        print("TEMPERATURES:", route["temperatures"])
        print("SELECTED DOCS:", route["selected_docs"])

    if route["status"] == "INSUFFICIENT_EVIDENCE":
        print("\nEvidence status: INSUFFICIENT_EVIDENCE")
        print(route["reason"])
        print("\nNo LLM call is made for unsupported post-harvest questions.")
        return {"route":route,"evidence":pd.DataFrame(),"context":""}

    retrieved=retrieve_postharvest(question,route["selected_docs"],k=k)
    verified=build_verified_context(route)
    print("\nEvidence status:", route["status"])
    print("\nVERIFIED DOCUMENT FACTS\n", verified)
    print("\nRELEVANT PDF PASSAGES")
    if retrieved.empty:
        print("No sufficiently relevant PDF passage was retrieved; verified profile only is available.")
    else:
        for _,r in retrieved.iterrows():
            preview=r.text[:900].replace("\n"," ")
            print(f"- {r.document_id} | page {r.page} | score={r.score:.3f}\n  {preview}...\n")
    return {"route":route,"evidence":retrieved,"context":verified}


In [65]:
# ============================================================
# Final unified JasmineGPT router — existing RAG remains intact
# ============================================================
def route_question_final(question):
    ql=question.lower()
    # Scheme questions can be handled by the existing scheme/module outside this notebook.
    # Here we only distinguish post-harvest from the pre-existing general RAG.
    if is_probably_postharvest(question):
        return "POSTHARVEST"
    return "GENERAL_RAG"


def ask_jasmine_gpt_final(question, k=5, conversation_id=None):
    route=route_question_final(question)
    if conversation_id:
        print(f"\n🌸 JasmineGPT | conversation={conversation_id} | ROUTE = {route}\n")
    else:
        print(f"\n🌸 JasmineGPT | ROUTE = {route}\n")

    if route == "POSTHARVEST":
        result=answer_postharvest(question,k=max(6,k),show_debug=True)
        if result["route"]["status"] == "INSUFFICIENT_EVIDENCE":
            print("\nFINAL ANSWER\nThe available post-harvest research does not provide sufficient evidence for this question.")
            return result

        # Optional LLM generation. The LLM is called only after evidence gating.
        api_key=os.environ.get("OPENROUTER_API_KEY","").strip()
        if not api_key:
            print("\nFINAL ANSWER\nEvidence was found, but OPENROUTER_API_KEY is not configured. The verified evidence above is the supported result.")
            return result

        passages="\n\n".join(
            f"[{r.document_id}, page {r.page}] {r.text}" for _,r in result["evidence"].iterrows()
        )
        prompt=f"""
You are JasmineGPT, an evidence-grounded assistant for jasmine farmers.
Use ONLY the verified document facts and retrieved PDF passages below.
Do not invent missing values. Distinguish controlled-study findings from universal recommendations.
State species/cultivar and experimental conditions when relevant.
If a question asks for a recommendation that the studies do not establish universally, say that clearly.

QUESTION:
{question}

VERIFIED DOCUMENT FACTS:
{result['context']}

RETRIEVED PDF PASSAGES:
{passages}

Write a concise farmer-friendly answer and cite the paper title / document ID used.
"""
        import requests
        resp=requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization":f"Bearer {api_key}","Content-Type":"application/json"},
            json={"model":"openai/gpt-4.1-mini","messages":[
                {"role":"system","content":"Use only supplied evidence. Never guess. Cite source document IDs."},
                {"role":"user","content":prompt}],
                "temperature":0.1,"max_tokens":700}, timeout=60)
        resp.raise_for_status()
        answer=resp.json()["choices"][0]["message"]["content"]
        print("\nFINAL ANSWER\n",answer)
        result["answer"]=answer
        return result

    # IMPORTANT: this calls your ORIGINAL search() function, untouched.
    retrieved = search(question, k)
    retrieved = retrieved[retrieved["score"] > 0.70]
    retrieved = retrieved.drop_duplicates(subset=["filename"])
    if len(retrieved)==0:
        print("The available jasmine research does not provide sufficient evidence.")
        return {"route":"GENERAL_RAG","retrieved":retrieved}

    print("Existing RAG evidence:")
    print(retrieved[[c for c in ["score","filename","category"] if c in retrieved.columns]].to_string(index=False))
    return {"route":"GENERAL_RAG","retrieved":retrieved}


## Integrated regression + post-harvest evaluation

The first block verifies the 10-query post-harvest test set used in the standalone v2 module. The second block checks that representative existing-RAG topics still enter `GENERAL_RAG`; their actual answer quality remains governed by the original corpus/index.


In [66]:
# ============================================================
# 10-query post-harvest acceptance test
# ============================================================
TESTS = [
    {"query":"Can I store Jasminum sambac at 7°C?", "expected_docs":["JAS-SAM-002"], "expected_status":"DIRECT_EVIDENCE"},
    {"query":"Can I store Jasminum sambac at 5°C?", "expected_docs":["JAS-SAM-004"], "expected_status":"DIRECT_EVIDENCE"},
    {"query":"What is passive MAP for Jasminum sambac storage?", "expected_docs":[], "expected_status":"INSUFFICIENT_EVIDENCE"},
    {"query":"Can I store Jasminum sambac at 2°C?", "expected_docs":[], "expected_status":"INSUFFICIENT_EVIDENCE"},
    {"query":"What temperatures were tested for Jasminum sambac storage?", "expected_docs":["JAS-SAM-002","JAS-SAM-004"], "expected_status":"MULTIPLE_STUDIES"},
    {"query":"What did the study report for 4% boric acid and 60 micron packaging?", "expected_docs":["JAS-SAM-002"], "expected_status":"DIRECT_EVIDENCE"},
    {"query":"How was gel ice used in jasmine export packaging?", "expected_docs":["JAS-SAM-001","JAS-SAM-005"], "expected_status":"MULTIPLE_STUDIES"},
    {"query":"How did mycelium foam perform during long-distance transport of Jasminum sambac?", "expected_docs":["JAS-SAM-005"], "expected_status":"DIRECT_EVIDENCE"},
    {"query":"What post-harvest treatment was studied for Pacha Mullai?", "expected_docs":["JAS-AUR-001"], "expected_status":"DIRECT_EVIDENCE"},
    {"query":"What is the best harvesting time for jasmine?", "expected_docs":[], "expected_status":"INSUFFICIENT_EVIDENCE"},
]

def normalize_docs(xs):
    return sorted(set(xs))

results=[]
for t in TESTS:
    r=route_postharvest(t["query"])
    dm=normalize_docs(r["selected_docs"])==normalize_docs(t["expected_docs"])
    sm=r["status"]==t["expected_status"]
    results.append({
        "query":t["query"],
        "expected_docs":",".join(t["expected_docs"]),
        "actual_docs":",".join(r["selected_docs"]),
        "expected_status":t["expected_status"],
        "actual_status":r["status"],
        "document_match":dm,
        "status_match":sm,
        "pass":dm and sm
    })

eval_df=pd.DataFrame(results)
eval_df.to_csv(PH_EVAL_PATH,index=False)
print(eval_df.to_string(index=False))
print(f"\nPost-harvest acceptance pass rate: {eval_df['pass'].mean()*100:.1f}%")


                                                                           query           expected_docs                                     actual_docs       expected_status         actual_status  document_match  status_match  pass
                                             Can I store Jasminum sambac at 7°C?             JAS-SAM-002                                     JAS-SAM-002       DIRECT_EVIDENCE       DIRECT_EVIDENCE            True          True  True
                                             Can I store Jasminum sambac at 5°C?             JAS-SAM-004                                     JAS-SAM-004       DIRECT_EVIDENCE       DIRECT_EVIDENCE            True          True  True
                                What is passive MAP for Jasminum sambac storage?                                                                         INSUFFICIENT_EVIDENCE INSUFFICIENT_EVIDENCE            True          True  True
                                             Can I store Jasminum sa

In [67]:
# ============================================================
# Existing-RAG routing regression checks (non-destructive)
# ============================================================
GENERAL_TESTS=[
    "How to control jasmine bud worm?",
    "What fertilizer is recommended for jasmine?",
    "How should I manage irrigation in jasmine?",
    "When should jasmine be pruned?",
]
reg=[]
for q in GENERAL_TESTS:
    reg.append({"query":q,"route":route_question_final(q)})
reg_df=pd.DataFrame(reg)
print(reg_df.to_string(index=False))
assert all(reg_df["route"]=="GENERAL_RAG"), "A general JasmineGPT query was incorrectly captured by post-harvest routing."
print("\nExisting-RAG routing regression: PASS")


                                      query       route
           How to control jasmine bud worm? GENERAL_RAG
What fertilizer is recommended for jasmine? GENERAL_RAG
 How should I manage irrigation in jasmine? GENERAL_RAG
             When should jasmine be pruned? GENERAL_RAG

Existing-RAG routing regression: PASS


In [68]:
# ============================================================
# Example live tests
# ============================================================
examples=[
    "what packaging was tested for gundumalli?",
    "Can I store Jasminum sambac at 7°C?",
    "What temperatures were tested for Jasminum sambac storage?",
    "What is passive MAP for Jasminum sambac storage?",
    "How to control jasmine bud worm?",
]
for q in examples:
    print("\n"+"#"*100)
    ask_jasmine_gpt_final(q, k=5)



####################################################################################################

🌸 JasmineGPT | ROUTE = POSTHARVEST

QUERY: what packaging was tested for gundumalli?
ROUTE: DIRECT_EVIDENCE
SPECIES: ['Jasminum sambac']
DOMAINS: ['packaging']
CONCEPTS: ['gundumalli']
TEMPERATURES: []
SELECTED DOCS: ['JAS-SAM-002']

Evidence status: DIRECT_EVIDENCE

VERIFIED DOCUMENT FACTS
 DOCUMENT JAS-SAM-002
Species: Jasminum sambac
Cultivar/ecotype: Gundumalli
Source note: Choudhury et al. (2019), direct J. sambac cv. Gundumalli packaging/shelf-life experiment.
Verified facts:
- Polythene bags were 20 × 12 cm, heat sealed, with no vents reported.
- The tested bag thicknesses included 40 µm and 60 µm.
- The best reported combination was 4% boric acid + 60 µm polythene + 7°C cold storage.
- Cold-room condition was 7°C with 80–85% RH.
- At 24 h for the best treatment, freshness was 98.75% and colour retention was 100%.

RELEVANT PDF PASSAGES
- JAS-SAM-002 | page 2 | score=0.673
  In

## Save integration manifest

The following artifacts are written under `metadata/final_postharvest_rag_v2/`. The original `metadata/faiss.index`, `metadata/chunks.csv`, `metadata/chunk_metadata.csv`, and original `search()` function are not modified by this new module.


In [ ]:
# ============================================================
# Save final integration manifest and configuration
# ============================================================
config={
    "embedding_model":EMBED_MODEL_NAME,
    "production_documents":POSTHARVEST_DOCS,
    "excluded_documents":EXCLUDED_DOCS,
    "postharvest_index":str(PH_INDEX_PATH),
    "postharvest_chunks":str(PH_CHUNKS_PATH),
    "postharvest_metadata":str(PH_META_PATH),
    "evaluation":str(PH_EVAL_PATH),
    "router": "POSTHARVEST -> separate v2 index; otherwise -> existing search()",
    "llm_policy": "LLM is not called when post-harvest evidence gate returns INSUFFICIENT_EVIDENCE",
    "verified_profiles": PROFILES
}
with open(PH_CONFIG_PATH,"w",encoding="utf-8") as f:
    json.dump(config,f,ensure_ascii=False,indent=2)

# Save production metadata with explicit status.
out_meta=ph_meta.copy()
out_meta["production_status"]="INCLUDED"
out_meta["excluded_reason"]=""
excl_rows=meta_all[meta_all["document_id"].isin(EXCLUDED_DOCS)].copy()
if not excl_rows.empty:
    excl_rows["production_status"]="EXCLUDED"
    excl_rows["excluded_reason"]=excl_rows["document_id"].map(EXCLUDED_DOCS).fillna("")
    out_meta=pd.concat([out_meta,excl_rows],ignore_index=True)
out_meta.to_csv(PH_META_PATH,index=False)

print("\nFINAL INTEGRATED ARTIFACTS")
for p in sorted(PH_OUT.iterdir()):
    print("-",p.name)


## How to use the final integrated notebook

Run the original notebook cells first, then run this complete integration section top-to-bottom. Your existing FAISS index is still used whenever the router returns `GENERAL_RAG`; post-harvest questions are intercepted before that search and use the separate post-harvest FAISS index.

**Important:** keep the five post-harvest PDFs and the verified metadata accessible in `Jasmine_Documents`. Keep Singh 2009 in your excluded/archive location; do not add it to the production document list.

For deployment, call `ask_jasmine_gpt_final(question, conversation_id=conversation_id)` instead of the old `ask_jasmine_gpt(question)` so that every question passes through the new router.


# 🌸 Phase 1 — Conversational Memory for JasmineGPT

This section is an **additive layer** on top of the existing JasmineGPT and the non-destructive post-harvest RAG. It does **not rebuild, overwrite, or merge** the existing FAISS index.

### What this adds
- Chat/session IDs
- Short-term conversation history
- Persistent history as JSON under Google Drive
- Follow-up question → standalone query rewriting
- Context carry-over for species/cultivar/topic
- Existing RAG regression protection
- Post-harvest routing still uses the verified 5-paper module
- No LLM call when post-harvest evidence is insufficient
- Automated multi-turn evaluation

### Data separation
```text
Scientific knowledge → FAISS / post-harvest FAISS
Conversation memory → JSON conversation store
Existing general RAG → existing search()
Post-harvest RAG → retrieve_postharvest() / answer_postharvest()
```


In [69]:
# ============================================================
# Phase 1.1 — Conversation-memory configuration
# ============================================================
from pathlib import Path
from datetime import datetime, timezone
import json, re, os, uuid, requests
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Jasmine_Documents")
MEMORY_DIR = ROOT / "metadata" / "conversation_memory_v1"
MEMORY_DIR.mkdir(parents=True, exist_ok=True)

MEMORY_FILE = MEMORY_DIR / "conversations.json"
MAX_TURNS_IN_CONTEXT = 8          # last 8 user+assistant messages
MAX_STORED_MESSAGES = 100         # per conversation

print("Conversation memory directory:", MEMORY_DIR)
print("Memory file:", MEMORY_FILE)


Conversation memory directory: /content/drive/MyDrive/Jasmine_Documents/metadata/conversation_memory_v1
Memory file: /content/drive/MyDrive/Jasmine_Documents/metadata/conversation_memory_v1/conversations.json


In [70]:
# ============================================================
# Phase 1.2 — Persistent conversation store
# ============================================================
def _utc_now():
    return datetime.now(timezone.utc).isoformat()


def load_conversations():
    if not MEMORY_FILE.exists():
        return {}
    try:
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception as e:
        print("Warning: could not load conversation store:", e)
        return {}


def save_conversations(store):
    tmp = MEMORY_FILE.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(store, f, ensure_ascii=False, indent=2)
    tmp.replace(MEMORY_FILE)


def new_conversation(title=None):
    store = load_conversations()
    conversation_id = "chat_" + uuid.uuid4().hex[:12]
    store[conversation_id] = {
        "conversation_id": conversation_id,
        "title": title or "New JasmineGPT Chat",
        "created_at": _utc_now(),
        "updated_at": _utc_now(),
        "messages": [],
        "memory": {
            "species": None,
            "cultivar": None,
            "domains": [],
            "last_route": None,
            "last_document_ids": []
        }
    }
    save_conversations(store)
    return conversation_id


def get_conversation(conversation_id):
    store = load_conversations()
    if conversation_id not in store:
        raise KeyError(f"Conversation not found: {conversation_id}")
    return store[conversation_id]


def append_message(conversation_id, role, content, metadata=None):
    store = load_conversations()
    if conversation_id not in store:
        raise KeyError(f"Conversation not found: {conversation_id}")
    msg = {
        "message_id": "msg_" + uuid.uuid4().hex[:12],
        "role": role,
        "content": str(content),
        "timestamp": _utc_now(),
        "metadata": metadata or {}
    }
    conv = store[conversation_id]
    conv["messages"].append(msg)
    conv["messages"] = conv["messages"][-MAX_STORED_MESSAGES:]
    conv["updated_at"] = _utc_now()
    save_conversations(store)
    return msg


def update_memory(conversation_id, **updates):
    store = load_conversations()
    conv = store[conversation_id]
    conv["memory"].update({k: v for k, v in updates.items() if v is not None})
    conv["updated_at"] = _utc_now()
    save_conversations(store)


def list_conversations(limit=20):
    store = load_conversations()
    rows=[]
    for cid, conv in store.items():
        rows.append({
            "conversation_id": cid,
            "title": conv.get("title"),
            "messages": len(conv.get("messages", [])),
            "updated_at": conv.get("updated_at")
        })
    return pd.DataFrame(rows).sort_values("updated_at", ascending=False).head(limit) if rows else pd.DataFrame()

print("Loaded conversations:", len(load_conversations()))


Loaded conversations: 14


In [71]:
# ============================================================
# Phase 1.3 — Conversation context extraction
# ============================================================
PHASE1_SPECIES = {
    "gundumalli": "Jasminum sambac",
    "ramanathapuram gundumalli": "Jasminum sambac",
    "jasminum sambac": "Jasminum sambac",
    "sambac": "Jasminum sambac",
    "pacha mullai": "Jasminum auriculatum",
    "jasminum auriculatum": "Jasminum auriculatum",
    "auriculatum": "Jasminum auriculatum",
}

PHASE1_CULTIVARS = {
    "gundumalli": "Gundumalli",
    "ramanathapuram gundumalli": "Ramanathapuram Gundumalli",
    "pacha mullai": "Pacha Mullai ecotype",
}

PHASE1_DOMAIN_TERMS = {
    "packaging": ["packaging", "package", "packing", "bag", "polythene", "polyethylene", "micron", "thermocol", "chitosan", "mycelium"],
    "storage": ["storage", "store", "cold storage", "temperature", "shelf life", "freshness"],
    "transportation": ["transport", "transportation", "long-distance", "export", "gel ice", "reefer"],
    "pest": ["pest", "pesticide", "insecticide", "bud worm", "thrips", "mite", "aphid", "whitefly"],
    "nutrition": ["fertilizer", "fertiliser", "npk", "nutrition", "fertigation"],
    "irrigation": ["irrigation", "drip", "water stress", "watering"],
    "pruning": ["pruning", "prune", "off season", "flowering", "bloom"],
    "disease": ["disease", "fungicide", "leaf spot", "virus", "pathogen"],
}

FOLLOWUP_MARKERS = re.compile(r"\b(it|this|that|these|those|they|them|same|also|again|what about|how long|how much|what size|which one|and then|there)\b", re.I)


def extract_entities(text):
    q = text.lower()
    species = None
    cultivar = None
    for alias, canonical in sorted(PHASE1_SPECIES.items(), key=lambda x: len(x[0]), reverse=True):
        if alias in q:
            species = canonical
            break
    for alias, canonical in sorted(PHASE1_CULTIVARS.items(), key=lambda x: len(x[0]), reverse=True):
        if alias in q:
            cultivar = canonical
            break
    domains=[]
    for d, terms in PHASE1_DOMAIN_TERMS.items():
        if any(term in q for term in terms):
            domains.append(d)
    return {"species":species, "cultivar":cultivar, "domains":domains}


def recent_messages(conversation_id, max_messages=MAX_TURNS_IN_CONTEXT):
    conv=get_conversation(conversation_id)
    return conv.get("messages", [])[-max_messages:]


def conversation_context_text(conversation_id, max_messages=MAX_TURNS_IN_CONTEXT):
    msgs=recent_messages(conversation_id, max_messages)
    return "\n".join(f"{m['role'].upper()}: {m['content']}" for m in msgs)


def get_memory_state(conversation_id):
    return get_conversation(conversation_id).get("memory", {})


def is_followup_question(question):
    q=question.strip()
    low=q.lower()
    if FOLLOWUP_MARKERS.search(q):
        return True
    # Very short contextual questions are often follow-ups.
    if len(q.split()) <= 7 and any(w in low for w in ["how", "what", "which", "where", "when", "can", "should"]):
        return True
    return False


In [72]:
# ============================================================
# Phase 1.4 — Standalone question resolver (deterministic first)
# ============================================================
# ============================================================
# Phase 1.4 — Improved standalone question resolver
# Deterministic first
# ============================================================

CONTEXT_REFERENCE_PATTERNS = [
    r"\bit\b",
    r"\bthis\b",
    r"\bthat\b",
    r"\bthese\b",
    r"\bthose\b",
    r"\bthe packaging\b",
    r"\bthe treatment\b",
    r"\bthe method\b",
    r"\bthe flowers\b",
    r"\bthe storage\b",
    r"\bthe temperature\b",
    r"\bthe bags?\b",
    r"\bthe results?\b",
    r"\bthe study\b",
    r"\bhow long did\b",
    r"\bwhat temperature\b",
    r"\bwhat packaging\b",
    r"\bhow was .* packaged\b",
    r"\bwas .* heat[- ]sealed\b",
    r"\bwas .* heat sealed\b",
    r"\bwhat was .* used\b",
    r"\bwhat was the\b",
    r"\bwhich .* does this\b",
    r"\bwhat about\b",
]


def looks_context_dependent(question: str) -> bool:
    """Detect questions that naturally depend on previous turns."""
    q = question.lower().strip()

    return any(
        re.search(pattern, q)
        for pattern in CONTEXT_REFERENCE_PATTERNS
    )


def resolve_followup_deterministic(question, conversation_id):
    conv = get_conversation(conversation_id)
    memory = conv.get("memory", {})
    q = question.strip()

    entities = extract_entities(q)

    # --------------------------------------------------------
    # 1. New conversation -> no rewrite needed
    # --------------------------------------------------------
    if not conv.get("messages"):
        return q, {
            "method": "none",
            "confidence": 1.0,
            "reason": "new conversation"
        }

    # --------------------------------------------------------
    # 2. Collect recent user messages
    # --------------------------------------------------------
    user_msgs = [
        m["content"]
        for m in conv.get("messages", [])
        if m.get("role") == "user"
    ]

    previous_user = user_msgs[-1] if user_msgs else ""
    previous_entities = extract_entities(previous_user)

    # --------------------------------------------------------
    # 3. Current explicit entities
    # --------------------------------------------------------
    explicit_species = entities.get("species") or []
    explicit_cultivar = entities.get("cultivar") or []
    explicit_domains = entities.get("domains") or []

    # --------------------------------------------------------
    # 4. Memory context
    # --------------------------------------------------------
    species = explicit_species or memory.get("species") or previous_entities.get("species") or []
    cultivar = explicit_cultivar or memory.get("cultivar") or previous_entities.get("cultivar") or []
    domains = explicit_domains or memory.get("domains") or previous_entities.get("domains") or []

    # --------------------------------------------------------
    # 5. IMPORTANT:
    # A question can contain a domain keyword and STILL be
    # context-dependent.
    #
    # Example:
    # "Was the packaging heat sealed?"
    # contains "packaging", but it refers to previous context.
    # --------------------------------------------------------
    context_dependent = looks_context_dependent(q)

    # --------------------------------------------------------
    # 6. Explicit standalone question
    #
    # Only accept as explicitly standalone when:
    # - it has a species/cultivar explicitly named, OR
    # - it is clearly not context-dependent and has enough
    #   explicit topic information.
    #
    # Domain alone is NOT sufficient.
    # --------------------------------------------------------
    if (
        not context_dependent
        and (explicit_species or explicit_cultivar)
    ):
        return q, {
            "method": "explicit",
            "confidence": 1.0,
            "reason": "question explicitly names the relevant species/cultivar"
        }

    # --------------------------------------------------------
    # 7. If the question is context-dependent, carry memory
    # --------------------------------------------------------
    parts = []

    if species:
        if isinstance(species, list):
            parts.extend(species)
        else:
            parts.append(species)

    if cultivar:
        if isinstance(cultivar, list):
            for c in cultivar:
                if c.lower() not in q.lower():
                    if c == "Pacha Mullai ecotype":
                        parts.append(c)
                    else:
                        parts.append(f"cv. {c}")
        else:
            if cultivar.lower() not in q.lower():
                if cultivar == "Pacha Mullai ecotype":
                    parts.append(cultivar)
                else:
                    parts.append(f"cv. {cultivar}")

    # --------------------------------------------------------
    # 8. Preserve original wording.
    # Add only missing scientific context.
    # --------------------------------------------------------
    rewritten = q

    if parts:
        missing_parts = [
            p for p in parts
            if p.lower() not in rewritten.lower()
        ]

        if missing_parts:
            rewritten = f"{rewritten} ({' '.join(missing_parts)})"

    # --------------------------------------------------------
    # 9. Add domain as routing context only.
    # This is NOT scientific evidence.
    # --------------------------------------------------------
    if domains:
        domain_values = domains if isinstance(domains, list) else [domains]

        missing_domains = [
            d for d in domain_values
            if d.lower() not in rewritten.lower()
        ]

        if missing_domains:
            rewritten = (
                f"{rewritten} "
                f"[context: {'; '.join(missing_domains)}]"
            )

    changed = rewritten != q

    # --------------------------------------------------------
    # 10. Return resolution metadata
    # --------------------------------------------------------
    if context_dependent and changed:
        reason = "context-dependent question; carried context from conversation memory"
        confidence = 0.97
        method = "deterministic_contextual"

    elif changed:
        reason = "carried relevant context from conversation memory"
        confidence = 0.95
        method = "deterministic"

    else:
        reason = "no safe rewrite needed"
        confidence = 0.75
        method = "deterministic"

    return rewritten, {
        "method": method,
        "confidence": confidence,
        "reason": reason
    }

def resolve_followup_llm(question, conversation_id):
    """Optional high-quality rewrite. Falls back safely to deterministic rewriting."""
    api_key=os.environ.get("OPENROUTER_API_KEY", "").strip()
    if not api_key:
        return resolve_followup_deterministic(question, conversation_id)
    context=conversation_context_text(conversation_id)
    memory=get_memory_state(conversation_id)
    prompt=f"""Rewrite the farmer's latest question as ONE standalone retrieval query.
Use only context explicitly present in the conversation. Do not add facts. Preserve species/cultivar/domain when they are clear. If the latest question is already standalone, return it unchanged.

Known memory: {json.dumps(memory, ensure_ascii=False)}
Conversation:
{context}

Latest question:
{question}

Return only the standalone query, no quotes, no explanation."""
    try:
        resp=requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization":f"Bearer {api_key}","Content-Type":"application/json"},
            json={"model":"openai/gpt-4.1-mini","messages":[
                {"role":"system","content":"You rewrite questions for retrieval. Never invent facts."},
                {"role":"user","content":prompt}],
                "temperature":0.0,"max_tokens":120}, timeout=30)
        resp.raise_for_status()
        rewritten=resp.json()["choices"][0]["message"]["content"].strip()
        if not rewritten:
            raise ValueError("empty rewrite")
        return rewritten, {"method":"llm", "confidence":0.98, "reason":"LLM contextual rewrite"}
    except Exception as e:
        fallback=resolve_followup_deterministic(question, conversation_id)
        fallback[1]["llm_error"]=str(e)
        return fallback


def resolve_query(question, conversation_id, use_llm_rewriter=False):
    # Always try deterministic resolution first
    rewritten, info = resolve_followup_deterministic(
        question,
        conversation_id
    )

    # High-confidence deterministic result -> keep it
    if info.get("confidence", 0) >= 0.90:
        return rewritten, info

    # Optional LLM fallback
    if use_llm_rewriter:
        return resolve_followup_llm(
            question,
            conversation_id
        )

    return rewritten, info


In [73]:
# ============================================================
# Phase 1.5 — Safe unified conversational chat wrapper
# ============================================================

def update_conversation_memory_from_turn(conversation_id, user_question, result):
    entities = extract_entities(user_question)
    existing = get_memory_state(conversation_id)

    species = entities["species"] or existing.get("species")
    cultivar = entities["cultivar"] or existing.get("cultivar")
    domains = entities["domains"] or existing.get("domains", [])

    route = result.get("route") if isinstance(result, dict) else None

    if isinstance(route, dict):
        if route.get("species"):
            species = species or route["species"][0]

        if route.get("domains"):
            domains = list(dict.fromkeys(
                domains + list(route.get("domains", []))
            ))

        last_docs = route.get("selected_docs", [])
        last_route = route.get("status")

    else:
        last_docs = []
        last_route = route

        if isinstance(result, dict) and "retrieved" in result:
            retrieved = result["retrieved"]
            if retrieved is not None and not retrieved.empty:
                last_docs = retrieved["filename"].tolist()[:5]

    update_memory(
        conversation_id,
        species=species,
        cultivar=cultivar,
        domains=domains,
        last_route=last_route,
        last_document_ids=last_docs
    )


def _print_history(conversation_id):
    conv = get_conversation(conversation_id)

    print("\n🧠 Conversation history")

    for m in conv.get("messages", [])[-12:]:
        role = "Farmer" if m["role"] == "user" else "JasmineGPT"
        print(f"{role}: {m['content']}")


def chat_once(
    conversation_id,
    question,
    k=5,
    use_llm_rewriter=False,
    show_history=True
):
    """
    One conversational turn.

    conversation_id is mandatory here because this function owns
    the multi-turn memory lifecycle.
    """

    question = question.strip()

    if not question:
        return {
            "conversation_id": conversation_id,
            "question": question,
            "standalone_query": "",
            "rewrite": {},
            "result": {"route": "EMPTY"}
        }

    # --------------------------------------------------------
    # 1. Always store the original farmer message first.
    # --------------------------------------------------------
    append_message(
        conversation_id,
        "user",
        question
    )

    # --------------------------------------------------------
    # 2. Profile / memory-only statement.
    #
    # Example:
    # "I grow Gundumalli jasmine."
    #
    # This MUST NOT trigger RAG retrieval.
    # --------------------------------------------------------
    if is_profile_statement(question):
        answer, info = handle_profile_statement(
            question,
            conversation_id
        )

        result = {
            "route": "MEMORY_UPDATE",
            "answer": answer,
            "memory": info.get("memory", {}),
            "profile_update": True
        }

        append_message(
            conversation_id,
            "assistant",
            answer,
            metadata={
                "route": "MEMORY_UPDATE",
                "memory_updated": True
            }
        )

        print("\n" + "=" * 88)
        print(f"🌸 JasmineGPT | conversation={conversation_id}")
        print("FARMER QUESTION:", question)
        print("ROUTE = MEMORY_UPDATE")
        print("ANSWER:", answer)
        print("MEMORY:", info.get("memory", {}))

        if show_history:
            _print_history(conversation_id)

        return {
            "conversation_id": conversation_id,
            "question": question,
            "standalone_query": question,
            "rewrite": {
                "method": "profile",
                "confidence": 1.0,
                "reason": "explicit farmer profile statement"
            },
            "result": result
        }

    # --------------------------------------------------------
    # 3. Resolve follow-up into a standalone retrieval query.
    # --------------------------------------------------------
    standalone, rewrite_info = resolve_query(
        question,
        conversation_id,
        use_llm_rewriter=use_llm_rewriter
    )

    print("\n" + "=" * 88)
    print(f"🌸 JasmineGPT | conversation={conversation_id}")
    print("FARMER QUESTION:", question)
    print("STANDALONE RETRIEVAL QUERY:", standalone)
    print("REWRITE:", rewrite_info)

    # --------------------------------------------------------
    # 4. Route to the correct knowledge module.
    #
    # conversation_id is passed through for traceability.
    # --------------------------------------------------------
    result = ask_jasmine_gpt_final(
        standalone,
        k=k,
        conversation_id=conversation_id
    )

    # --------------------------------------------------------
    # 5. Preserve traceability in chat history.
    # --------------------------------------------------------
    route_label = (
        result.get("route")
        if isinstance(result, dict)
        else None
    )

    metadata = {
        "standalone_query": standalone,
        "rewrite": rewrite_info,
        "route": route_label,
    }

    answer_text = (
        result.get("answer")
        if isinstance(result, dict) and result.get("answer")
        else "Evidence/retrieval result generated above; see notebook output."
    )

    append_message(
        conversation_id,
        "assistant",
        answer_text,
        metadata=metadata
    )

    update_conversation_memory_from_turn(
        conversation_id,
        question,
        result
    )

    if show_history:
        _print_history(conversation_id)

    return {
        "conversation_id": conversation_id,
        "question": question,
        "standalone_query": standalone,
        "rewrite": rewrite_info,
        "result": result
    }


def start_chat(title=None):
    cid = new_conversation(title=title)
    print("Created conversation:", cid)
    return cid


def reset_chat(conversation_id):
    store = load_conversations()

    if conversation_id in store:
        del store[conversation_id]
        save_conversations(store)
        print("Deleted:", conversation_id)
    else:
        print("Conversation not found:", conversation_id)


## Phase 1 safety rule

The conversation layer **does not become a source of evidence**. It is used only to resolve references such as **“it”**, **“same method”**, **“what about 5°C?”**, or an omitted cultivar/species. Scientific claims still come from the existing JasmineGPT RAG or the verified post-harvest module.

For an unsupported post-harvest question, the existing evidence gate remains authoritative and the LLM is not called.


In [74]:
# ============================================================
# Phase 1.6 — Multi-turn regression/evaluation
# ============================================================

def run_phase1_test_case(
    title,
    turns,
    expected_last_route=None
):
    cid = start_chat(title)
    traces = []

    for q in turns:
        out = chat_once(
            cid,
            q,
            show_history=False
        )
        traces.append(out)

    last = traces[-1]

    route = None
    if isinstance(last.get("result"), dict):
        route = last["result"].get("route")

    passed = (
        expected_last_route is None
        or route == expected_last_route
    )

    return {
        "title": title,
        "conversation_id": cid,
        "turns": len(turns),
        "last_original": turns[-1],
        "last_standalone": last["standalone_query"],
        "last_route": route,
        "pass": passed
    }


PHASE1_CASES = [
    {
        "title": "Profile statement does not trigger RAG",
        "turns": [
            "I grow Gundumalli jasmine."
        ],
        "expected_last_route": "MEMORY_UPDATE"
    },
    {
        "title": "Gundumalli packaging follow-up",
        "turns": [
            "I grow Gundumalli jasmine.",
            "What packaging was tested for it?"
        ],
        "expected_last_route": "POSTHARVEST"
    },
    {
        "title": "Heat-sealed packaging follow-up",
        "turns": [
            "I grow Gundumalli jasmine.",
            "What packaging was tested?",
            "Was the packaging heat sealed?"
        ],
        "expected_last_route": "POSTHARVEST"
    },
    {
        "title": "Gundumalli storage follow-up",
        "turns": [
            "I am asking about Gundumalli.",
            "Can I store it at 7°C?"
        ],
        "expected_last_route": "POSTHARVEST"
    },
    {
        "title": "Post-harvest result follow-up",
        "turns": [
            "What packaging was tested for Gundumalli?",
            "What were the bag thicknesses?"
        ],
        "expected_last_route": "POSTHARVEST"
    },
    {
        "title": "Pacha Mullai switch",
        "turns": [
            "I have Pacha Mullai jasmine.",
            "What post-harvest treatment was studied?"
        ],
        "expected_last_route": "POSTHARVEST"
    },
    {
        "title": "Existing pest RAG preserved",
        "turns": [
            "I have a jasmine bud worm problem.",
            "Which pesticide research is relevant?"
        ],
        "expected_last_route": "GENERAL_RAG"
    },
    {
        "title": "Unsupported harvest topic stays blocked",
        "turns": [
            "I want advice about jasmine post-harvest handling.",
            "What is the best harvesting time?"
        ],
        "expected_last_route": "POSTHARVEST"
    },
]


phase1_results = []

for case in PHASE1_CASES:
    r = run_phase1_test_case(
        case["title"],
        case["turns"],
        case["expected_last_route"]
    )
    phase1_results.append(r)


phase1_eval = pd.DataFrame(phase1_results)

print(
    phase1_eval[
        [
            "title",
            "last_original",
            "last_standalone",
            "last_route",
            "pass"
        ]
    ].to_string(index=False)
)

print(
    f"\nPhase 1 routing pass rate: "
    f"{phase1_eval['pass'].mean() * 100:.1f}%"
)

PHASE1_EVAL_PATH = (
    MEMORY_DIR / "phase1_conversational_evaluation.csv"
)

phase1_eval.to_csv(
    PHASE1_EVAL_PATH,
    index=False
)

print("Saved evaluation:", PHASE1_EVAL_PATH)


Created conversation: chat_28cccf74ec4a

🌸 JasmineGPT | conversation=chat_28cccf74ec4a
FARMER QUESTION: I grow Gundumalli jasmine.
STANDALONE RETRIEVAL QUERY: I grow Gundumalli jasmine.
REWRITE: {'method': 'explicit', 'confidence': 1.0, 'reason': 'question explicitly names the relevant species/cultivar'}

🌸 JasmineGPT | ROUTE = GENERAL_RAG

Existing RAG evidence:
   score                                                                                                                 filename    category
0.735650                                                                                                              TH-4900.pdf Cultivation
0.733456                                                                                                                D3665.pdf Cultivation
0.720367 Influence of different pruning months and pruning height on growth and flowering of gundumalli (Jasminum sambac. L.).pdf     Pruning
0.719957                                                          

In [75]:
# ============================================================
# Phase 1.7 — Inspect persistent chat history
# ============================================================
print("\nConversation list")
display(list_conversations(20))



Conversation list


,conversation_id,title,messages,updated_at
19,chat_d52bae354a37,Unsupported harvest topic stays blocked,4,2026-09-12T07:52:27.626582+00:00
18,chat_3357f5a43cd3,Existing pest RAG preserved,4,2026-09-12T07:52:27.400424+00:00
17,chat_a267e3699338,Pacha Mullai switch,4,2026-09-12T07:52:27.038849+00:00
16,chat_36c684867f0e,Post-harvest result follow-up,4,2026-09-12T07:52:26.774754+00:00
15,chat_30d1da3c5bb7,Gundumalli storage follow-up,4,2026-09-12T07:52:23.027038+00:00
14,chat_28cccf74ec4a,Gundumalli packaging follow-up,4,2026-09-12T07:52:20.426553+00:00
13,chat_a5391e0fdc7d,JasmineGPT Phase 1 Demo,10,2026-09-12T07:44:06.602990+00:00
12,chat_cccf70168d46,Unsupported harvest topic stays blocked,4,2026-09-12T07:43:26.983469+00:00
11,chat_f7f71ebf9b82,Existing pest RAG preserved,4,2026-09-12T07:43:26.746906+00:00
10,chat_f00444a486de,Pacha Mullai switch,4,2026-09-12T07:43:26.407555+00:00


In [76]:
# ============================================================
# Example live tests — conversational interface
# ============================================================

conversation_id = start_chat("JasmineGPT Phase 1 Demo")

test_questions = [
    "I grow Gundumalli jasmine.",
    "What packaging was tested?",
    "Was the packaging heat sealed?",
    "What temperature was used?",
    "How long did it last?",
]

for q in test_questions:
    chat_once(
        conversation_id,
        q,
        show_history=True
    )


Created conversation: chat_359cf363cdc0

Enter farmer questions. Type 'exit' to stop.

Farmer: I grow Gundumalli jasmine.

🌸 JasmineGPT | conversation=chat_359cf363cdc0
FARMER QUESTION: I grow Gundumalli jasmine.
STANDALONE RETRIEVAL QUERY: I grow Gundumalli jasmine.
REWRITE: {'method': 'explicit', 'confidence': 1.0, 'reason': 'question explicitly names the relevant species/cultivar'}

🌸 JasmineGPT | ROUTE = GENERAL_RAG

Existing RAG evidence:
   score                                                                                                                 filename    category
0.735650                                                                                                              TH-4900.pdf Cultivation
0.733456                                                                                                                D3665.pdf Cultivation
0.720367 Influence of different pruning months and pruning height on growth and flowering of gundumalli (Jasminum sambac. L.).

## How to use this module in your final application

Use one `conversation_id` for the lifetime of one chat session:

```python
conversation_id = start_chat("Farmer Chat")

result = chat_once(
    conversation_id,
    "I grow Gundumalli jasmine."
)

result = chat_once(
    conversation_id,
    "What packaging was tested?"
)
```

The conversation layer then performs:

```text
farmer message
      ↓
profile/memory-only check
      ↓
contextual question resolver
      ↓
domain router
      ↓
existing RAG OR post-harvest RAG
      ↓
grounded result
      ↓
save answer + metadata to conversation history
```

The original `search()` and existing FAISS index remain unchanged. The post-harvest FAISS remains separate. Conversation history is stored separately from scientific knowledge.

For a direct non-conversational query, you may still call:

```python
ask_jasmine_gpt_final("your standalone question", k=5)
```

For conversational use, prefer `chat_once(conversation_id, question)`.


# ✅ Phase 1.1 Integration Notes

The final Phase 1 flow intentionally keeps responsibilities separate:

- `conversation_id` belongs to the conversation layer.
- `chat_once(conversation_id, question)` owns memory and multi-turn state.
- `ask_jasmine_gpt_final(..., conversation_id=...)` receives the session ID for traceability but does not change the existing retrieval logic.
- `search()` and the original FAISS index are untouched.
- Post-harvest retrieval remains a separate index.
- Explicit farmer profile statements update memory without triggering RAG.
- Generic statements such as "I have a jasmine bud worm problem" are **not** treated as profile statements and continue to the existing pest RAG.
